## Carbon System Analysis
Saving carbon system output

In [1]:
import xarray as xr
import gsw
import PyCO2SYS as pyco2
import numpy as np
import pandas as pd

## 5 year mean

In [9]:
# load map data

path='/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results'

base = xr.open_dataset('/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/forthalweg_physics202111_20180101_20221231.nc').mean('time')
basechem = xr.open_dataset('/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/forthalweg_chemistry202111_20180101_20221231.nc').mean('time')
basebio = xr.open_dataset('/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/forthalweg_biology202111_20180101_20221231.nc').mean('time')

# All 
files = [f'{path}/All_{year}main_primarytracers_forthalweg_{year}0101_{year}1231.nc' for year in range(2018,2022+1)]
all = xr.open_mfdataset(files).mean('time')

# Salish 
files = [f'{path}/Salish_{year}main_primarytracers_forthalweg_monthly_{year}0101_{year}1231.nc' for year in range(2018,2022+1)]
salish = xr.open_mfdataset(files).mean('time')

Y,depth,X =np.meshgrid(all.gridY.values,all.depth.values,all.gridX.values)
lat,lon = np.full(np.shape(Y),49),np.full(np.shape(Y),-123)

In [2]:
dims = ("depth", "gridY", "gridX")

In [ ]:
# Base Case Projection

# convert to correct units
SP = gsw.conversions.SP_from_SR(base.vosaline.values)
SA = gsw.conversions.SA_from_SP(SP,depth,lon,lat)
rho = gsw.density.rho(SA,base.votemper.values,depth)
TA = basechem.total_alkalinity.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
DIC = basechem.dissolved_inorganic_carbon.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
Temp = gsw.conversions.t_from_CT(SA,base.votemper.values,depth)

# mask where 0
SP = np.where(base.vosaline.values==0,np.nan,SP)
Temp = np.where(base.vosaline.values==0,np.nan,Temp)
TA = np.where(base.vosaline.values==0,np.nan,TA)
DIC = np.where(base.vosaline.values==0,np.nan,DIC)

baseco2 = pyco2.sys(
    par1=TA,          # TA [µmol/kg]
    par2=DIC,          # DIC [µmol/kg]
    par1_type=1,        # Total alkalinity
    par2_type=2,        # Dissolved inorganic carbon
    salinity=SP,      # Practical salinity
    temperature=Temp,    # In situ temperature [°C]
    pressure=depth,       # Seawater pressure [dbar]
    opt_k_carbonic=10,  # Lueker et al. (2000)
)

ds = xr.Dataset({
    key: (dims, np.asarray(value))
    for key, value in baseco2.items()
    if np.asarray(value).ndim == len(dims)
    and np.asarray(value).shape == np.shape(TA)
})

ds.to_netcdf("/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/arag_basecase_5yr.nc")


In [17]:
# All Cases Projection

# convert to correct units
SP = gsw.conversions.SP_from_SR(all.vosaline.values)
SA = gsw.conversions.SA_from_SP(SP,depth,lon,lat)
rho = gsw.density.rho(SA,all.votemper.values,depth)
TA = all.total_alkalinity.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
DIC = all.dissolved_inorganic_carbon.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
Temp = gsw.conversions.t_from_CT(SA,all.votemper.values,depth)

# mask where 0
SP = np.where(all.vosaline.values==0,np.nan,SP)
Temp = np.where(all.vosaline.values==0,np.nan,Temp)
TA = np.where(all.vosaline.values==0,np.nan,TA)
DIC = np.where(all.vosaline.values==0,np.nan,DIC)

allco2 = pyco2.sys(
    par1=TA,          # TA [µmol/kg]
    par2=DIC,          # DIC [µmol/kg]
    par1_type=1,        # Total alkalinity
    par2_type=2,        # Dissolved inorganic carbon
    salinity=SP,      # Practical salinity
    temperature=Temp,    # In situ temperature [°C]
    pressure=depth,       # Seawater pressure [dbar]
    opt_k_carbonic=10,  # Lueker et al. (2000)
)

ds = xr.Dataset({
    key: (dims, np.asarray(value))
    for key, value in allco2.items()
    if np.asarray(value).ndim == len(dims)
    and np.asarray(value).shape == np.shape(TA)
})

ds.to_netcdf("/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/arag_allcase_5yr.nc")


In [10]:
# salish Case Projection

# convert to correct units
SP = gsw.conversions.SP_from_SR(salish.vosaline.values)
SA = gsw.conversions.SA_from_SP(SP,depth,lon,lat)
rho = gsw.density.rho(SA,salish.votemper.values,depth)
TA = salish.total_alkalinity.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
DIC = salish.dissolved_inorganic_carbon.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
Temp = gsw.conversions.t_from_CT(SA,salish.votemper.values,depth)

# mask where 0
SP = np.where(salish.vosaline.values==0,np.nan,SP)
Temp = np.where(salish.vosaline.values==0,np.nan,Temp)
TA = np.where(salish.vosaline.values==0,np.nan,TA)
DIC = np.where(salish.vosaline.values==0,np.nan,DIC)

salishco2 = pyco2.sys(
    par1=TA,          # TA [µmol/kg]
    par2=DIC,          # DIC [µmol/kg]
    par1_type=1,        # Total alkalinity
    par2_type=2,        # Dissolved inorganic carbon
    salinity=SP,      # Practical salinity
    temperature=Temp,    # In situ temperature [°C]
    pressure=depth,       # Seawater pressure [dbar]
    opt_k_carbonic=10,  # Lueker et al. (2000)
)

ds = xr.Dataset({
    key: (dims, np.asarray(value))
    for key, value in salishco2.items()
    if np.asarray(value).ndim == len(dims)
    and np.asarray(value).shape == np.shape(TA)
})

ds.to_netcdf("/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/arag_salishcase_5yr.nc")

## seasonal

In [11]:
# load map data

path='/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results'

base = xr.open_dataset('/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/forthalweg_monthly_physics202111_20180101_20221231.nc')
basechem = xr.open_dataset('/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/forthalweg_monthly_chemistry202111_20180101_20221231.nc')

# All 
files = [f'{path}/All_{year}main_primarytracers_forthalweg_monthly_{year}0101_{year}1231.nc' for year in range(2018,2022+1)]
all = xr.open_mfdataset(files)

# Salish 
files = [f'{path}/Salish_{year}main_primarytracers_forthalweg_monthly_{year}0101_{year}1231.nc' for year in range(2018,2022+1)]
salish = xr.open_mfdataset(files)

Y,depth,X =np.meshgrid(all.gridY.values,all.depth.values,all.gridX.values)
lat,lon = np.full(np.shape(Y),49),np.full(np.shape(Y),-123)

In [19]:
# All Cases Projection - winter (dec,jan,feb)

winterbool= (pd.to_datetime(all.time.values).month <=2) | (pd.to_datetime(all.time.values).month ==12)
_,winterbool,_,_ = np.meshgrid(all.depth.values,winterbool,all.gridY.values,all.gridX.values)
allwinter = all.where(winterbool).mean('time')

# convert to correct units
SP = gsw.conversions.SP_from_SR(allwinter.vosaline.values)
SA = gsw.conversions.SA_from_SP(SP,depth,lon,lat)
rho = gsw.density.rho(SA,allwinter.votemper.values,depth)
TA = allwinter.total_alkalinity.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
DIC = allwinter.dissolved_inorganic_carbon.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
Temp = gsw.conversions.t_from_CT(SA,allwinter.votemper.values,depth)

# mask where 0
SP = np.where(allwinter.vosaline.values==0,np.nan,SP)
Temp = np.where(allwinter.vosaline.values==0,np.nan,Temp)
TA = np.where(allwinter.vosaline.values==0,np.nan,TA)
DIC = np.where(allwinter.vosaline.values==0,np.nan,DIC)

allwinterco2 = pyco2.sys(
    par1=TA,          # TA [µmol/kg]
    par2=DIC,          # DIC [µmol/kg]
    par1_type=1,        # Total alkalinity
    par2_type=2,        # Dissolved inorganic carbon
    salinity=SP,      # Practical salinity
    temperature=Temp,    # In situ temperature [°C]
    pressure=depth,       # Seawater pressure [dbar]
    opt_k_carbonic=10,  # Lueker et al. (2000)
)

ds = xr.Dataset({
    key: (dims, np.asarray(value))
    for key, value in allwinterco2.items()
    if np.asarray(value).ndim == len(dims)
    and np.asarray(value).shape == np.shape(TA)
})

ds.to_netcdf("/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/arag_allcase_winter.nc")


In [8]:
# All Cases Projection - summer (jun,jul,aug)

summerbool= (pd.to_datetime(all.time.values).month >=6) & (pd.to_datetime(all.time.values).month <=8)
_,summerbool,_,_ = np.meshgrid(all.depth.values,summerbool,all.gridY.values,all.gridX.values)
allsummer = all.where(summerbool).mean('time')

# convert to correct units
SP = gsw.conversions.SP_from_SR(allsummer.vosaline.values)
SA = gsw.conversions.SA_from_SP(SP,depth,lon,lat)
rho = gsw.density.rho(SA,allsummer.votemper.values,depth)
TA = allsummer.total_alkalinity.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
DIC = allsummer.dissolved_inorganic_carbon.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
Temp = gsw.conversions.t_from_CT(SA,allsummer.votemper.values,depth)

# mask where 0
SP = np.where(allsummer.vosaline.values==0,np.nan,SP)
Temp = np.where(allsummer.vosaline.values==0,np.nan,Temp)
TA = np.where(allsummer.vosaline.values==0,np.nan,TA)
DIC = np.where(allsummer.vosaline.values==0,np.nan,DIC)

allsummerco2 = pyco2.sys(
    par1=TA,          # TA [µmol/kg]
    par2=DIC,          # DIC [µmol/kg]
    par1_type=1,        # Total alkalinity
    par2_type=2,        # Dissolved inorganic carbon
    salinity=SP,      # Practical salinity
    temperature=Temp,    # In situ temperature [°C]
    pressure=depth,       # Seawater pressure [dbar]
    opt_k_carbonic=10,  # Lueker et al. (2000)
)

ds = xr.Dataset({
    key: (dims, np.asarray(value))
    for key, value in allsummerco2.items()
    if np.asarray(value).ndim == len(dims)
    and np.asarray(value).shape == np.shape(TA)
})

ds.to_netcdf("/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/arag_allcase_summer.nc")


In [21]:
# Base Case Projection - winter (dec,jan,feb)

winterbool= (pd.to_datetime(base.time.values).month <=2) | (pd.to_datetime(base.time.values).month ==12)
_,winterbool,_,_ = np.meshgrid(base.depth.values,winterbool,base.gridY.values,base.gridX.values)
basewinter = base.where(winterbool).mean('time')
basechemwinter = basechem.where(winterbool).mean('time')

# convert to correct units
SP = gsw.conversions.SP_from_SR(basewinter.vosaline.values)
SA = gsw.conversions.SA_from_SP(SP,depth,lon,lat)
rho = gsw.density.rho(SA,basewinter.votemper.values,depth)
TA = basechemwinter.total_alkalinity.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
DIC = basechemwinter.dissolved_inorganic_carbon.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
Temp = gsw.conversions.t_from_CT(SA,basewinter.votemper.values,depth)

# mask where 0
SP = np.where(basewinter.vosaline.values==0,np.nan,SP)
Temp = np.where(basewinter.vosaline.values==0,np.nan,Temp)
TA = np.where(basewinter.vosaline.values==0,np.nan,TA)
DIC = np.where(basewinter.vosaline.values==0,np.nan,DIC)

basewinterco2 = pyco2.sys(
    par1=TA,          # TA [µmol/kg]
    par2=DIC,          # DIC [µmol/kg]
    par1_type=1,        # Total alkalinity
    par2_type=2,        # Dissolved inorganic carbon
    salinity=SP,      # Practical salinity
    temperature=Temp,    # In situ temperature [°C]
    pressure=depth,       # Seawater pressure [dbar]
    opt_k_carbonic=10,  # Lueker et al. (2000)
)

ds = xr.Dataset({
    key: (dims, np.asarray(value))
    for key, value in basewinterco2.items()
    if np.asarray(value).ndim == len(dims)
    and np.asarray(value).shape == np.shape(TA)
})

ds.to_netcdf("/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/arag_basecase_winter.nc")

In [7]:
# Base Case Projection - summer (jun,jul,aug)

summerbool= (pd.to_datetime(base.time.values).month >=6) & (pd.to_datetime(base.time.values).month <=8)
_,summerbool,_,_ = np.meshgrid(base.depth.values,summerbool,base.gridY.values,base.gridX.values)
basesummer = base.where(summerbool).mean('time')
basechemsummer = basechem.where(summerbool).mean('time')

# convert to correct units
SP = gsw.conversions.SP_from_SR(basesummer.vosaline.values)
SA = gsw.conversions.SA_from_SP(SP,depth,lon,lat)
rho = gsw.density.rho(SA,basesummer.votemper.values,depth)
TA = basechemsummer.total_alkalinity.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
DIC = basechemsummer.dissolved_inorganic_carbon.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
Temp = gsw.conversions.t_from_CT(SA,basesummer.votemper.values,depth)

# mask where 0
SP = np.where(basesummer.vosaline.values==0,np.nan,SP)
Temp = np.where(basesummer.vosaline.values==0,np.nan,Temp)
TA = np.where(basesummer.vosaline.values==0,np.nan,TA)
DIC = np.where(basesummer.vosaline.values==0,np.nan,DIC)

basesummerco2 = pyco2.sys(
    par1=TA,          # TA [µmol/kg]
    par2=DIC,          # DIC [µmol/kg]
    par1_type=1,        # Total alkalinity
    par2_type=2,        # Dissolved inorganic carbon
    salinity=SP,      # Practical salinity
    temperature=Temp,    # In situ temperature [°C]
    pressure=depth,       # Seawater pressure [dbar]
    opt_k_carbonic=10,  # Lueker et al. (2000)
)

ds = xr.Dataset({
    key: (dims, np.asarray(value))
    for key, value in basesummerco2.items()
    if np.asarray(value).ndim == len(dims)
    and np.asarray(value).shape == np.shape(TA)
})

ds.to_netcdf("/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/arag_basecase_summer.nc")

In [17]:
# Salish Case Projection - winter (dec,jan,feb)

winterbool= (pd.to_datetime(salish.time.values).month <=2) | (pd.to_datetime(salish.time.values).month ==12)
_,winterbool,_,_ = np.meshgrid(salish.depth.values,winterbool,salish.gridY.values,salish.gridX.values)
salishwinter = salish.where(winterbool).mean('time')

# convert to correct units
SP = gsw.conversions.SP_from_SR(salishwinter.vosaline.values)
SA = gsw.conversions.SA_from_SP(SP,depth,lon,lat)
rho = gsw.density.rho(SA,salishwinter.votemper.values,depth)
TA = salishwinter.total_alkalinity.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
DIC = salishwinter.dissolved_inorganic_carbon.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
Temp = gsw.conversions.t_from_CT(SA,salishwinter.votemper.values,depth)

# mask where 0
SP = np.where(salishwinter.vosaline.values==0,np.nan,SP)
Temp = np.where(salishwinter.vosaline.values==0,np.nan,Temp)
TA = np.where(salishwinter.vosaline.values==0,np.nan,TA)
DIC = np.where(salishwinter.vosaline.values==0,np.nan,DIC)

salishwinterco2 = pyco2.sys(
    par1=TA,          # TA [µmol/kg]
    par2=DIC,          # DIC [µmol/kg]
    par1_type=1,        # Total alkalinity
    par2_type=2,        # Dissolved inorganic carbon
    salinity=SP,      # Practical salinity
    temperature=Temp,    # In situ temperature [°C]
    pressure=depth,       # Seawater pressure [dbar]
    opt_k_carbonic=10,  # Lueker et al. (2000)
)

ds = xr.Dataset({
    key: (dims, np.asarray(value))
    for key, value in salishwinterco2.items()
    if np.asarray(value).ndim == len(dims)
    and np.asarray(value).shape == np.shape(TA)
})

ds.to_netcdf("/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/arag_salishcase_winter.nc")

In [6]:
# Salish Case Projection - summer (dec,jan,feb)

summerbool= (pd.to_datetime(salish.time.values).month >=6) & (pd.to_datetime(salish.time.values).month <=8)
_,summerbool,_,_ = np.meshgrid(salish.depth.values,summerbool,salish.gridY.values,salish.gridX.values)
salishsummer = salish.where(summerbool).mean('time')

# convert to correct units
SP = gsw.conversions.SP_from_SR(salishsummer.vosaline.values)
SA = gsw.conversions.SA_from_SP(SP,depth,lon,lat)
rho = gsw.density.rho(SA,salishsummer.votemper.values,depth)
TA = salishsummer.total_alkalinity.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
DIC = salishsummer.dissolved_inorganic_carbon.values * (1/rho) * 1000 # convert from mmol/m3 to umol/kg
Temp = gsw.conversions.t_from_CT(SA,salishsummer.votemper.values,depth)

# mask where 0
SP = np.where(salishsummer.vosaline.values==0,np.nan,SP)
Temp = np.where(salishsummer.vosaline.values==0,np.nan,Temp)
TA = np.where(salishsummer.vosaline.values==0,np.nan,TA)
DIC = np.where(salishsummer.vosaline.values==0,np.nan,DIC)

salishsummerco2 = pyco2.sys(
    par1=TA,          # TA [µmol/kg]
    par2=DIC,          # DIC [µmol/kg]
    par1_type=1,        # Total alkalinity
    par2_type=2,        # Dissolved inorganic carbon
    salinity=SP,      # Practical salinity
    temperature=Temp,    # In situ temperature [°C]
    pressure=depth,       # Seawater pressure [dbar]
    opt_k_carbonic=10,  # Lueker et al. (2000)
)

ds = xr.Dataset({
    key: (dims, np.asarray(value))
    for key, value in salishsummerco2.items()
    if np.asarray(value).ndim == len(dims)
    and np.asarray(value).shape == np.shape(TA)
})

ds.to_netcdf("/ocean/rbeutel/MOAD/analysis-becca/projections/summary_results/arag_salishcase_summer.nc")